# Módulo 2: AgentCore Runtime — O Primeiro Deploy do Agente

![Overview](../shared/img/02.drawio.png)

Neste módulo, você vai subir a Aria de verdade para rodar no **Amazon Bedrock AgentCore Runtime**.

## O que você vai aprender

- **AgentCore Runtime** — a camada de hospedagem da AWS que roda e escala o seu agente de forma gerenciada
- **Como o código é empacotado** — saindo do seu computador até virar um contêiner lá na nuvem
- **Como funciona o streaming** — como as perguntas e respostas vão e voltam em tempo real (como se a IA estivesse digitando)

Até o final desta aula, a Aria estará viva e respondendo perguntas (mas ainda sem ferramentas extras nem memória).

---
## Atualizando as dependências

O código abaixo garante que tudo que precisamos dos módulos anteriores já esteja pronto. Se você pulou algum passo, ele arruma a casa pra você.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared.ensure_ready import ensure_ready

# Verifica se todos os pré-requisitos deste módulo estão prontos.
config = ensure_ready("02")

---
## Conhecendo o código do agente

O código da Aria mora no arquivo `agent/main.py`. Antes de subir ele, vamos entender os pontos principais.

### A Classe BedrockAgentCoreApp

Todo agente do AgentCore começa instanciando a classe **`BedrockAgentCoreApp`**. É ela que lida com as requisições HTTP, verifica se está tudo funcionando e fala com os controles do Runtime.

### O decorador `@app.entrypoint`

É com esse decorador que a gente avisa onde o código principal começa. Ele recebe duas coisas:
- **`payload`** — a requisição chegando (com a mensagem do usuário, ID da sessão, etc)
- **`context`** — uns metadados adicionais opcionais

### O padrão assíncrono (Streaming)

A função principal é um **gerador assíncrono**. Em vez de devolver tudo de uma vez no final, ela vai soltando a resposta aos pouquinhos (com a palavra `yield`). Isso é o que dá aquele efeito da IA escrevendo o texto na tela em tempo real.

### O que o AgentCore Runtime faz na infraestrutura?

Quando você executa o deploy do seu agente no **AgentCore Runtime**, a infraestrutura AWS gerencia automaticamente a seguinte cadeia de execução:
1. **Containerização e Imagem Canonical**: O código é empacotado em uma imagem OCI/Docker otimizada para a arquitetura ARM64 (AWS Graviton).
2. **MicroVMs Efêmeras (Firecracker)**: Para cada nova sessão (`runtimeSessionId`), o AgentCore Runtime instancia uma microVM dedicada e isolada em nível de hipervisor KVM usando a tecnologia open-source **Firecracker**. Isso garante zero contaminação de memória ou estado entre diferentes usuários.
3. **Auto-scaling e Ciclo de Vida**: A microVM permanece ativa durante a sessão do usuário (reaproveitando instâncias `_agent` para baixíssima latência) e é automaticamente destruída após inatividade, garantindo o modelo pay-per-use.
4. **Roteamento Server-Sent Events (SSE)**: As respostas são transmitidas token a token utilizando o protocolo SSE sobre HTTP/2 streaming.

### A grande vantagem: Isolamento Nativo de Sessão

Em arquiteturas tradicionais, o desenvolvedor precisa gerenciar manualmente o estado de sessão em bancos como Redis ou DynamoDB. No **AgentCore Runtime**, **cada sessão roda em sua própria microVM Firecracker**. O que isso proporciona na prática?

- **Gerenciamento de Estado Simplificado**: A instância do agente (`_agent`) vive na memória do processo da microVM durante o ciclo de vida da sessão.
- **Preservação de Histórico no Processo**: O histórico de mensagens é retido em memória local enquanto a microVM estiver ativa.
- **Segurança Rígida de Multitenancy**: Isolamento de hardware total entre clientes distintos. Ao encerrar a sessão, o ambiente é desalocado e sanitizado completamente.

A AWS abstrai toda a complexidade de orquestração de servidores, permitindo foco exclusivo na engenharia de IA.

Para ver o código completinho, abra o arquivo [agent/main.py](agent/main.py) em outra aba.

As bibliotecas que ele precisa usar estão listadas no [agent/requirements.txt](agent/requirements.txt).

---
## Subindo o Agente pra nuvem (Deploy)

O caminho do deploy é bem direto:

1. **Empacotar o Código** — a pasta é compactada num arquivo ZIP
2. **Upload pro S3** — mandamos esse arquivo para o Amazon S3
3. **Criação do Runtime** — O AgentCore baixa o ZIP, cria o contêiner e sobe ele no ar
4. **Status READY** — A IA tá no ar e pronta pra receber mensagens

Toda essa etapa burocrática foi escondida numa funçãozinha de ajuda chamada `deploy_agent` pra gente ganhar tempo.

> **E no mundo real?** No dia a dia de trabalho, em vez desse script, você provavelmente usaria o AWS CDK ou o CLI oficial do AgentCore pra subir o agente.

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import deploy_agent

# Módulo que empacota o código do agente e faz o deploy no AgentCore Runtime.
result = deploy_agent.deploy(
    # Caminho para o diretório com o código do agente (main.py + requirements.txt).
    agent_dir="agent",
    runtime_name="aria_agent",
    # clean_start: Apaga qualquer Runtime antigo antes de criar um novo (evita conflitos).
    clean_start=True,
)

runtime_arn = result["runtime_arn"]
print(f"\nRuntime ARN: {runtime_arn}")

---
## Habilitar Logs de Observabilidade (Tracing)

Antes de bater papo com a Aria, vamos ativar a função que rastreia tudo o que acontece lá dentro (Tracing), pra gente poder acompanhar depois no CloudWatch.

1. Abra o painel do **Amazon Bedrock AgentCore** no seu navegador e vá em **Runtimes**
2. Selecione o runtime **aria_agent**
3. Desça até a seção de **Tracing** e clique em **Edit** (Editar)

![Tracing section](../shared/img/tracing-01.png)

4. Marque a chave **Enable** (Habilitar) e clique em **Save** (Salvar)

![Enable tracing](../shared/img/tracing-02.png)

> **Nota:** Essa opção precisa ser ativada manualmente para cada serviço. Vamos fazer isso de novo mais na frente nos outros módulos.

---
## Testando o agente (Invocação)

A Aria tá no ar! Vamos chamar a moça usando a API crua da AWS (`boto3`). Aqui não tem atalho, você vai ver o código oficial de como se comunica com a IA.

Você manda um JSON com a chave `prompt`, e a AWS te devolve uma resposta em um monte de eventos fracionados (**Server-Sent Events**). Vamos ver a resposta crua e feia primeiro pra entender o que tá acontecendo.

In [ ]:
import boto3
import json
import uuid

client = boto3.client("bedrock-agentcore", region_name="us-east-1")

response = client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=str(uuid.uuid4()),
    contentType="application/json",
    accept="text/event-stream",
    payload=json.dumps({"prompt": "Hello! What can you do?"}).encode("utf-8"),
)

for line in response["response"].iter_lines():
    if line:
        print(line.decode("utf-8"), flush=True)

### Entendendo a resposta crua

A resposta que chega é cheia de marcações, com linhas começando com `data: ` contendo um JSON cada uma. O texto real do que a Aria falou fica escondido aqui dentro:

```
event → contentBlockDelta → delta → text
```

Vamos fazer um pequeno parser (extrator) em Python para limpar toda a sujeira e puxar só o texto puro:

In [ ]:
# Invoke again, this time parsing the SSE stream for clean output
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import utils

response = client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=str(uuid.uuid4()),
    payload=json.dumps({"prompt": "Hello! What can you do?"}).encode(),
)

utils.stream_sse_response(response["response"])

---
## Bate-papo pelo Terminal (Interativo)

Legal, agora que vimos como funciona por debaixo dos panos, vamos usar um aplicativo simples de chat direto no terminal pra facilitar nossas conversas com a Aria no resto do curso.

Abra um **terminal**:
- No VS Code: **Terminal > Novo Terminal**
- Entre na pasta deste módulo

E rode:

```bash
cd /workshop/02-runtime
python ../shared/chat.py
```

Tente brincar com ela fazendo as seguintes perguntas:

1. `Olá! O que você consegue fazer?`
2. `Qual a raiz quadrada de 7.293.461?` — (Ela provavelmente vai inventar um número porque não sabe calcular isso de cabeça)
3. `Qual é o clima de Seattle hoje?` — (Ela não sabe, pois não tem acesso à internet... ainda)
4. Diga `Meu nome é Alex` — logo depois mande: `Qual é o meu nome?` — (Ela vai lembrar porque a sessão da microVM segura essa memória)
5. Digite `new` para resetar o chat e pergunte `Qual é meu nome?` — (Ela já esqueceu tudo)
6. Digite `quit` pra sair

> **Dica:** O script do chat usa essa mesmíssima função que acabamos de fazer. Ele pega as configurações automaticamente. Daqui a pouco vamos até incluir uma opção de segurança com `--auth`.

Se você tentou fazer as perguntas, deve ter notado:

- **Ela é péssima de matemática** — Se mandar ela fazer conta difícil, ela apenas 'inventa' a resposta.
- **Ela está isolada do mundo** — Ela não sabe ler o clima do dia, preços da bolsa ou buscar links.
- **Amnésia** — Assim que fecha a janela, ela esquece tudo o que você conversou.

---
## O que vem a seguir

Aria tá viva, mas só tem o cérebro cru treinado de fábrica. Ela não consegue rodar programação, não navega na internet e é esquecida.

No **Módulo 3: Ferramentas**, vamos dar dois superpoderes novos pra ela:
- **Interpretador de Código** — Uma maquininha isolada para ela programar em Python, calcular com perfeição, gerar gráficos e limpar arquivos.
- **Navegador Web** — Um navegador invisível (Chrome) para ela abrir sites reais, pesquisar informações e extrair textos da web na hora.

Tudo isso fornecido de forma nativa e gerenciada pela AWS AgentCore. Você não vai precisar criar nenhuma infraestrutura pra isso rodar!

---
## Record progress

In [ ]:
# Adiciona o diretório pai ao PATH do Python para permitir imports relativos.
import sys; sys.path.insert(0, '..')
# Importa as ferramentas compartilhadas do workshop.
from shared import progress

progress.show("02")

---

**Próximo Passo: [Módulo 3 -- Dando Ferramentas para a Aria](../03-tools/notebook.ipynb)**